# 02 — Generation (M5)

Runs conditions A … C-Mask on a **GPU** (T4, 16 GB): one model in memory at a time, finished records skipped on re-run.

* **dev** any time (prompt development). B-Abs and APG need the dev thresholds from notebook 01 part 2 and are skipped until they exist.
* **test** only after M8 has written `outputs/frozen.json`; the code refuses otherwise.
* Mistral is gated: accept its terms on Hugging Face and add `HF_TOKEN` as a secret (Colab: *Secrets*; Kaggle: *Add-ons → Secrets*).

In [ ]:
# Setup: works on Colab and locally (a GPU is needed to actually generate).
import os, sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Yashas-BR/grounded-product-description.git"
    CODE_DIR = "/content/ghd"
    from google.colab import drive
    drive.mount("/content/drive")
    if os.path.isdir(CODE_DIR):
        !git -C {CODE_DIR} pull --ff-only
    else:
        !git clone {REPO_URL} {CODE_DIR}
    os.chdir(CODE_DIR)
    !pip install -q -r requirements.txt
    CONFIGS = []
else:
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
    CONFIGS = ["configs/local.yaml"]
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import subprocess
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], check=True)

In [ ]:
import json, logging
import groundedgen  # noqa: F401  (import order fix on Windows)
from groundedgen import utils

logging.basicConfig(level=logging.INFO)
cfg = utils.load_config(extra_paths=CONFIGS)
utils.apply_runtime_env(cfg)
utils.set_seed(cfg["seed"])
utils.load_hf_token()
env = utils.capture_env()
print(json.dumps(env, indent=2))
utils.update_run_info(utils.project_path(cfg, "outputs", "run_info.json"), {"env": env})

## Dev run

Re-run this cell after a disconnect: it resumes where it stopped.

In [ ]:
from groundedgen import generate

summary = generate.run_matrix(cfg, "dev")
print(json.dumps(summary, indent=2))
# M5 acceptance: peak GPU memory under 15 GB for every model.
assert all(v is None or v < 15 for v in summary["peak_gpu_gb"].values()), summary["peak_gpu_gb"]

In [ ]:
# Inspect a few outputs per condition.
import pandas as pd
rows = []
for path in sorted(utils.project_path(cfg, "outputs", "generation").rglob("*.jsonl")):
    for r in utils.read_jsonl(path)[:2]:
        rows.append({"condition": r["condition"], "generator": r["generator"].split("/")[-1],
                     "extractor": (r["extractor"] or "").split("/")[-1], "evidence": r["evidence"],
                     "text": r["text"]})
pd.set_option("display.max_colwidth", 200)
pd.DataFrame(rows)

## Test run (after M8 only)

`generate.run_matrix(cfg, "test")` raises unless `outputs/frozen.json` matches the current prompts and thresholds.